# Day 4 - Evaluation and Grad-CAM

Today you will evaluate your **four saved models**, inspect confusion matrices
and per-class precision/recall/F1, and review **20 correct + 20 incorrect**
predictions from the selected fine-tuned ResNet18. No training is needed today.

We use **validation** errors to choose a proposed improvement. The held-out test
set remains reserved for the final Day 5 comparison.

1. Keep your latest **`day3_backup (1).zip`** ready.
2. Run cells in order. Upload that ZIP when prompted. A T4 GPU helps; CPU also works.
3. **Pause at the review form:** inspect each image, choose two categories, and
   click **Save and next** until the counter reaches **40/40**.
4. Fill the two observation answers, then create and download your backup.
5. Download the executed notebook from the top **File -> Download -> Download .ipynb** menu.

Avoid Run all: the review form needs your input before the final cells can finish.

## 1. Check packages

Only missing packages are installed, preserving Colab's supplied torch/torchvision
pair. There is no need to download ImageNet weights again: your checkpoints contain them.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

modules = {"Pillow": "PIL", "numpy": "numpy", "matplotlib": "matplotlib",
           "torch": "torch", "torchvision": "torchvision", "tensorboard": "tensorboard",
           "ipywidgets": "ipywidgets"}
missing = [package for package, module in modules.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
import torch
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
current = Path.cwd()
ROOT = Path("/content/flower-cnn") if IN_COLAB else (current.parent if current.name == "notebooks" else current)
ROOT.mkdir(parents=True, exist_ok=True)
print("PyTorch:", torch.__version__)
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (supported; slower)")

## 2. Load the helpers included in this notebook

This cell includes the model definitions, evaluation code, and review form.
Your backup supplies the saved data splits and trained model checkpoints.

In [ ]:
# @title Load the reusable project files
HELPER_SOURCES = {'flower_data.py': '"""Day 1: verified download, image audit, and reproducible split manifests.\n\nRun: python flower_data.py --root .\nOnly Pillow is needed. No training or image modification happens here.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\nimport random\nimport shutil\nimport tarfile\nimport tempfile\nimport urllib.request\nfrom collections import Counter, defaultdict\nfrom pathlib import Path, PurePosixPath\n\nfrom PIL import Image, ImageDraw, ImageOps, UnidentifiedImageError\n\nDATASET_URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"\nARCHIVE_SHA256 = "4c54ace7911aaffe13a365c34f650e71dd5bf1be0a58b464e5a7183e3e595d9c"\nARCHIVE_BYTES = 228813984\nCHECKSUM_SOURCE = "https://github.com/tensorflow/datasets/blob/master/tensorflow_datasets/datasets/tf_flowers/checksums.tsv"\nCLASS_NAMES = ("daisy", "dandelion", "roses", "sunflowers", "tulips")\nSPLITS = ("train", "val", "test")\nMANIFEST_FIELDS = ("relative_path", "class_name", "label", "width", "height", "file_sha256", "pixel_sha256", "split")\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef download_dataset(raw_dir: Path) -> Path:\n    """Verify the official archive before extracting regular files only."""\n    raw_dir = Path(raw_dir).resolve()\n    raw_dir.mkdir(parents=True, exist_ok=True)\n    dataset_dir = raw_dir / "flower_photos"\n    marker = dataset_dir / ".verified_archive_sha256"\n    if dataset_dir.exists():\n        if marker.is_file() and marker.read_text().strip() == ARCHIVE_SHA256:\n            return dataset_dir\n        raise ValueError(f"Unverified existing directory: {dataset_dir}. Use a new --root.")\n\n    archive = raw_dir / "flower_photos.tgz"\n    if not archive.exists():\n        partial = archive.with_suffix(".tgz.part")\n        received, next_notice = 0, 20 * 1024 * 1024\n        print(f"Downloading {ARCHIVE_BYTES / 1024**2:.1f} MiB of flower images...", flush=True)\n        request = urllib.request.Request(DATASET_URL, headers={"User-Agent": "flower-cnn-day1/1.0"})\n        with urllib.request.urlopen(request, timeout=60) as response, partial.open("wb") as out:\n            for block in iter(lambda: response.read(1024 * 1024), b""):\n                out.write(block)\n                received += len(block)\n                if received >= next_notice:\n                    print(f"  {received / 1024**2:.0f} MiB downloaded", flush=True)\n                    next_notice += 20 * 1024 * 1024\n        if partial.stat().st_size != ARCHIVE_BYTES or sha256_file(partial) != ARCHIVE_SHA256:\n            partial.unlink(missing_ok=True)\n            raise ValueError("Dataset download failed its size/SHA-256 check; rerun to retry.")\n        partial.replace(archive)\n    if archive.stat().st_size != ARCHIVE_BYTES or sha256_file(archive) != ARCHIVE_SHA256:\n        raise ValueError(f"Cached archive checksum mismatch: {archive}. Rename it and retry.")\n\n    # Staging keeps a partially extracted dataset from being mistaken for a complete one.\n    with tempfile.TemporaryDirectory(prefix="extract-", dir=raw_dir) as temp:\n        staging = Path(temp).resolve()\n        with tarfile.open(archive, "r:gz") as source:\n            for member in source:\n                parts = PurePosixPath(member.name).parts\n                if not parts or parts[0] != "flower_photos" or ".." in parts or "\\\\" in member.name:\n                    raise ValueError(f"Unexpected archive path: {member.name}")\n                target = staging.joinpath(*parts).resolve()\n                if not target.is_relative_to(staging):\n                    raise ValueError(f"Archive path escapes staging: {member.name}")\n                if member.isdir():\n                    target.mkdir(parents=True, exist_ok=True)\n                elif member.isfile():\n                    target.parent.mkdir(parents=True, exist_ok=True)\n                    with source.extractfile(member) as incoming, target.open("wb") as out:\n                        shutil.copyfileobj(incoming, out)\n                else:\n                    raise ValueError(f"Unsupported archive member: {member.name}")\n        extracted = staging / "flower_photos"\n        if not all((extracted / name).is_dir() for name in CLASS_NAMES):\n            raise ValueError("Archive does not contain all five flower categories.")\n        (extracted / ".verified_archive_sha256").write_text(ARCHIVE_SHA256 + "\\n")\n        extracted.rename(dataset_dir)\n    return dataset_dir\n\n\ndef audit_images(dataset_dir: Path) -> tuple[list[dict], dict]:\n    """Decode every image; keep one copy per exact decoded RGB image.\n\n    Identical images with conflicting class labels are all excluded for review.\n    This detects exact duplicates, not crops or other near duplicates.\n    """\n    candidates, corrupt = [], []\n    dataset_dir = Path(dataset_dir)\n    for label, class_name in enumerate(CLASS_NAMES):\n        class_dir = dataset_dir / class_name\n        if not class_dir.is_dir():\n            raise ValueError(f"Missing class directory: {class_dir}")\n        for path in sorted(class_dir.iterdir()):\n            if not path.is_file() or path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:\n                continue\n            relative = path.relative_to(dataset_dir).as_posix()\n            try:\n                with Image.open(path) as image:\n                    rgb = ImageOps.exif_transpose(image).convert("RGB")\n                    rgb.load()\n                    width, height = rgb.size\n                    pixels = hashlib.sha256(f"{width}x{height}:RGB:".encode() + rgb.tobytes()).hexdigest()\n                candidates.append({"relative_path": relative, "class_name": class_name, "label": label,\n                                   "width": width, "height": height, "file_sha256": sha256_file(path),\n                                   "pixel_sha256": pixels})\n            except (OSError, ValueError, UnidentifiedImageError, Image.DecompressionBombError) as error:\n                corrupt.append({"relative_path": relative, "error": str(error)})\n\n    groups = defaultdict(list)\n    for row in candidates:\n        groups[row["pixel_sha256"]].append(row)\n    clean, duplicates, conflicts = [], [], []\n    for group in groups.values():\n        if len({row["label"] for row in group}) > 1:\n            conflicts.append([row["relative_path"] for row in group])\n            continue\n        clean.append(group[0])\n        duplicates.extend({"kept": group[0]["relative_path"], "excluded": row["relative_path"]} for row in group[1:])\n    clean.sort(key=lambda row: row["relative_path"])\n    if any(not any(row["class_name"] == name for row in clean) for name in CLASS_NAMES):\n        raise ValueError("Every class must contain readable, non-conflicting images.")\n    audit = {"images_scanned": len(candidates) + len(corrupt), "readable_images": len(candidates),\n             "usable_images": len(clean), "corrupt_images": corrupt, "excluded_duplicates": duplicates,\n             "conflicting_label_groups": conflicts, "duplicate_detection": "Exact decoded RGB pixels; near duplicates are not detected."}\n    return clean, audit\n\n\ndef stratified_split(rows: list[dict], seed: int = 42) -> list[dict]:\n    """Assign approximately 70/15/15 percent within each class, deterministically."""\n    grouped = defaultdict(list)\n    for row in rows:\n        grouped[row["class_name"]].append(dict(row))\n    result = []\n    for class_name in sorted(grouped):\n        group = sorted(grouped[class_name], key=lambda row: row["relative_path"])\n        if len(group) < 7:\n            raise ValueError(f"Need at least 7 usable images in {class_name} for three splits.")\n        random.Random(f"{seed}:{class_name}").shuffle(group)\n        n_val = int(len(group) * 0.15 + 0.5)\n        n_test = int(len(group) * 0.15 + 0.5)\n        n_train = len(group) - n_val - n_test\n        for index, row in enumerate(group):\n            row["split"] = "train" if index < n_train else "val" if index < n_train + n_val else "test"\n            result.append(row)\n    return sorted(result, key=lambda row: row["relative_path"])\n\n\ndef validate_splits(rows: list[dict]) -> None:\n    if not rows:\n        raise ValueError("Split manifest cannot be empty.")\n    if len({row["relative_path"] for row in rows}) != len(rows):\n        raise ValueError("A file appears more than once in the split manifest.")\n    if len({row["pixel_sha256"] for row in rows}) != len(rows):\n        raise ValueError("Duplicate decoded images remain in the split manifest.")\n    for row in rows:\n        if row["split"] not in SPLITS:\n            raise ValueError(f"Unknown split: {row[\'split\']}")\n        if row["class_name"] not in CLASS_NAMES or int(row["label"]) != CLASS_NAMES.index(row["class_name"]):\n            raise ValueError("Class name and numeric label do not match.")\n    for name in CLASS_NAMES:\n        if {row["split"] for row in rows if row["class_name"] == name} != set(SPLITS):\n            raise ValueError(f"Class {name} is missing from a split.")\n\n\ndef save_sample_grid(rows: list[dict], dataset_dir: Path, destination: Path, seed: int = 42) -> None:\n    """Only training images are shown; labels are folder labels, not predictions."""\n    tile_width, tile_height = 190, 166\n    canvas = Image.new("RGB", (5 * tile_width, 5 * tile_height + 40), "#f5f7fb")\n    draw = ImageDraw.Draw(canvas)\n    draw.text((12, 12), "Training samples: five images per class", fill="#17253c")\n    for class_index, class_name in enumerate(CLASS_NAMES):\n        group = [row for row in rows if row["class_name"] == class_name and row["split"] == "train"]\n        selection = random.Random(f"{seed}:{class_name}:preview").sample(group, min(5, len(group)))\n        for column, row in enumerate(selection):\n            x, y = column * tile_width + 8, class_index * tile_height + 40\n            with Image.open(dataset_dir / row["relative_path"]) as image:\n                thumbnail = ImageOps.fit(ImageOps.exif_transpose(image).convert("RGB"), (174, 134))\n                canvas.paste(thumbnail, (x, y))\n            draw.text((x, y + 138), class_name, fill="#17253c")\n    canvas.save(destination)\n\n\ndef write_json(path: Path, value) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, indent=2) + "\\n", encoding="utf-8")\n\n\ndef record_environment(destination: Path) -> None:\n    packages = {}\n    for name in ("Pillow", "numpy", "matplotlib", "torch", "torchvision"):\n        try:\n            packages[name] = importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:\n            packages[name] = None\n    write_json(destination, {"python": platform.python_version(), "platform": platform.platform(), "packages": packages})\n\n\ndef prepare_project(root: Path, seed: int = 42, dataset_dir: Path | None = None) -> dict:\n    root = Path(root).resolve()\n    dataset_dir = Path(dataset_dir).resolve() if dataset_dir else download_dataset(root / "data" / "raw")\n    print("Auditing images and checking exact duplicates...", flush=True)\n    rows, audit = audit_images(dataset_dir)\n    rows = stratified_split(rows, seed)\n    validate_splits(rows)\n    processed, reports = root / "data" / "processed", root / "outputs" / "day1"\n    processed.mkdir(parents=True, exist_ok=True)\n    reports.mkdir(parents=True, exist_ok=True)\n    manifest = processed / "splits.csv"\n    with manifest.open("w", newline="", encoding="utf-8") as stream:\n        writer = csv.DictWriter(stream, fieldnames=MANIFEST_FIELDS)\n        writer.writeheader()\n        writer.writerows(rows)\n    class_counts = {name: {split: sum(row["class_name"] == name and row["split"] == split for row in rows)\n                           for split in SPLITS} for name in CLASS_NAMES}\n    summary = {"dataset": "Flower Photos", "dataset_url": DATASET_URL, "source_archive_sha256": ARCHIVE_SHA256 if dataset_dir == root / "data" / "raw" / "flower_photos" else None,\n               "dataset_dir": str(dataset_dir), "seed": seed, "split_ratios": {"train": 0.7, "val": 0.15, "test": 0.15},\n               "class_to_idx": {name: label for label, name in enumerate(CLASS_NAMES)},\n               "class_counts": class_counts, "split_counts": dict(Counter(row["split"] for row in rows)),\n               "width_range": [min(row["width"] for row in rows), max(row["width"] for row in rows)],\n               "height_range": [min(row["height"] for row in rows), max(row["height"] for row in rows)],\n               "manifest_sha256": sha256_file(manifest), "audit": audit}\n    write_json(reports / "data_summary.json", summary)\n    write_json(reports / "audit.json", audit)\n    write_json(processed / "class_to_idx.json", summary["class_to_idx"])\n    record_environment(reports / "environment.json")\n    save_sample_grid(rows, dataset_dir, reports / "sample_grid.jpg", seed)\n    lines = ["# Day 1 data audit", "", f"Usable images: {len(rows)}. Seed: {seed}.", "",\n             "| Class | Train | Validation | Test |", "|---|---:|---:|---:|"]\n    lines += [f"| {name} | {counts[\'train\']} | {counts[\'val\']} | {counts[\'test\']} |" for name, counts in class_counts.items()]\n    lines += ["", f"Unreadable files: {len(audit[\'corrupt_images\'])}.",\n              f"Excluded exact duplicates: {len(audit[\'excluded_duplicates\'])}.",\n              f"Conflicting-label groups excluded: {len(audit[\'conflicting_label_groups\'])}.", "",\n              "Manual label review and near-duplicate review are still needed. The class folders provide the labels.",\n              "Only training images appear in sample_grid.jpg. Reuse splits.csv for every experiment.",\n              "No model has been trained; there are no accuracy results yet."]\n    (reports / "data_report.md").write_text("\\n".join(lines) + "\\n", encoding="utf-8")\n    print(json.dumps({"usable_images": len(rows), "split_counts": summary["split_counts"], "report": str(reports / "data_report.md")}, indent=2), flush=True)\n    return summary\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--root", type=Path, default=Path(__file__).resolve().parent)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--dataset-dir", type=Path, help="Use an already downloaded folder containing the five class directories.")\n    args = parser.parse_args()\n    prepare_project(args.root, args.seed, args.dataset_dir)\n', 'flower_pipeline.py': '"""PyTorch datasets and separate transforms for the saved Day 1 splits."""\n\nfrom __future__ import annotations\n\nimport csv\nimport json\nimport os\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom PIL import Image, ImageOps\nfrom torch.utils.data import DataLoader, Dataset\nfrom torchvision import transforms\n\nfrom flower_data import CLASS_NAMES, sha256_file, validate_splits\n\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\nIMAGENET_STD = (0.229, 0.224, 0.225)\n\n\nclass FlowerDataset(Dataset):\n    def __init__(self, dataset_dir: Path, rows: list[dict], transform):\n        self.dataset_dir = Path(dataset_dir).resolve()\n        self.rows = rows\n        self.transform = transform\n        for row in rows:\n            image_path = (self.dataset_dir / row["relative_path"]).resolve()\n            if not image_path.is_relative_to(self.dataset_dir) or not image_path.is_file():\n                raise ValueError(f"Missing or invalid image path: {image_path}")\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, index):\n        row = self.rows[index]\n        with Image.open(self.dataset_dir / row["relative_path"]) as source:\n            image = ImageOps.exif_transpose(source).convert("RGB")\n        return self.transform(image), int(row["label"])\n\n\ndef seed_worker(worker_id):\n    worker_seed = torch.initial_seed() % 2**32\n    np.random.seed(worker_seed)\n    random.seed(worker_seed)\n\n\ndef seed_everything(seed=42):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n\n\ndef make_loaders(root: Path, batch_size=32, num_workers=0, augment=False, seed=42, dataset_dir=None,\n                 crop_scale=(0.75, 1.0)):\n    """Baseline training is unaugmented; val/test are always deterministic.\n\n    Start a fresh set of loaders with the same seed for each experiment.\n    Fixed seeds improve repeatability; identical GPU results across machines\n    and library versions are not guaranteed.\n    """\n    root = Path(root).resolve()\n    if len(crop_scale) != 2 or not 0 < crop_scale[0] <= crop_scale[1] <= 1:\n        raise ValueError("Crop scale must contain increasing area fractions in (0, 1].")\n    manifest = root / "data" / "processed" / "splits.csv"\n    summary = json.loads((root / "outputs" / "day1" / "data_summary.json").read_text())\n    if sha256_file(manifest) != summary["manifest_sha256"]:\n        raise ValueError("Split manifest changed since preparation. Restore it or rerun data preparation.")\n    with manifest.open(newline="", encoding="utf-8") as stream:\n        rows = list(csv.DictReader(stream))\n    validate_splits(rows)\n    dataset_dir = Path(dataset_dir) if dataset_dir else root / "data" / "raw" / "flower_photos"\n    normalise = [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]\n    evaluation_transform = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), *normalise])\n    train_transform = transforms.Compose([\n        transforms.RandomResizedCrop(224, scale=tuple(crop_scale)),\n        transforms.RandomHorizontalFlip(),\n        transforms.RandomRotation(15),\n        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),\n        *normalise,\n    ]) if augment else evaluation_transform\n    seed_everything(seed)\n    loaders = {}\n    for split in ("train", "val", "test"):\n        dataset = FlowerDataset(dataset_dir, [row for row in rows if row["split"] == split],\n                                train_transform if split == "train" else evaluation_transform)\n        generator = torch.Generator().manual_seed(seed)\n        loaders[split] = DataLoader(dataset, batch_size=batch_size, shuffle=split == "train",\n                                    num_workers=num_workers, pin_memory=torch.cuda.is_available(),\n                                    worker_init_fn=seed_worker, generator=generator, drop_last=False)\n    return loaders, list(CLASS_NAMES)\n\n\ndef save_eda_charts(root: Path):\n    """Standard matplotlib charts of counts and original image dimensions."""\n    root = Path(root)\n    config = root / "outputs" / ".matplotlib"\n    config.mkdir(parents=True, exist_ok=True)\n    os.environ.setdefault("MPLCONFIGDIR", str(config.resolve()))\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n\n    with (root / "data" / "processed" / "splits.csv").open(newline="", encoding="utf-8") as stream:\n        rows = list(csv.DictReader(stream))\n    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), layout="constrained")\n    bottom = np.zeros(len(CLASS_NAMES))\n    for split, colour in zip(("train", "val", "test"), ("#2962d9", "#e3a11a", "#249977")):\n        counts = [sum(row["class_name"] == name and row["split"] == split for row in rows) for name in CLASS_NAMES]\n        axes[0].bar(CLASS_NAMES, counts, bottom=bottom, color=colour, label=split)\n        bottom += np.array(counts)\n    axes[0].set(title="Images per class and split", ylabel="Number of images")\n    axes[0].legend()\n    axes[1].hist([int(row["width"]) for row in rows], bins=25, alpha=0.65, label="Width", color="#2962d9")\n    axes[1].hist([int(row["height"]) for row in rows], bins=25, alpha=0.6, label="Height", color="#249977")\n    axes[1].set(title="Original image dimensions", xlabel="Pixels", ylabel="Number of images")\n    axes[1].legend()\n    destination = root / "outputs" / "day1" / "eda_charts.png"\n    fig.savefig(destination, dpi=150)\n    plt.close(fig)\n    return destination\n', 'flower_training.py': '"""Day 2: custom CNN and paired training/augmentation experiments.\n\nRun: python flower_training.py --root . --epochs 12\nFull training requires CUDA unless --allow-cpu is explicitly supplied.\nBounded CPU checks use --smoke and are saved separately from full experiments.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nimport platform\nimport time\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path, PurePosixPath\nfrom itertools import islice\n\nimport torch\nfrom torch import nn\n\nfrom flower_data import CLASS_NAMES, download_dataset, record_environment, sha256_file, validate_splits, write_json\nfrom flower_pipeline import make_loaders, seed_everything\n\nEXPECTED_DAY1_MANIFEST_SHA256 = "4fc79e62cb179ff75a18772ba16f72e091184263e6fb8f4731819bbdc07c2dbb"\nDAY1_REQUIRED = ("data/processed/splits.csv", "data/processed/class_to_idx.json", "outputs/day1/data_summary.json")\n\n\ndef restore_day1_backup(backup: Path, root: Path) -> dict:\n    """Restore known data artifacts only; never execute Python from an archive."""\n    root = Path(root).resolve()\n    with zipfile.ZipFile(backup) as archive:\n        names = archive.namelist()\n        for name in DAY1_REQUIRED:\n            if names.count(name) != 1:\n                raise ValueError(f"Backup must contain exactly one {name}.")\n        selected = [info for info in archive.infolist()\n                    if not info.is_dir() and info.filename.startswith(("data/processed/", "outputs/day1/"))]\n        if any(info.file_size > 20 * 1024**2 for info in selected) or sum(info.file_size for info in selected) > 40 * 1024**2:\n            raise ValueError("Day 1 backup artifacts are unexpectedly large.")\n        contents = {info.filename: archive.read(info) for info in selected}\n        for name in contents:\n            if ".." in PurePosixPath(name).parts or "\\\\" in name:\n                raise ValueError(f"Invalid backup path: {name}")\n        summary = json.loads(contents["outputs/day1/data_summary.json"])\n        manifest_bytes = contents["data/processed/splits.csv"]\n        actual = hashlib.sha256(manifest_bytes).hexdigest()\n        if actual != EXPECTED_DAY1_MANIFEST_SHA256 or actual != summary["manifest_sha256"]:\n            raise ValueError("Backup splits differ from the verified Day 1 splits. Keep the original manifest.")\n        validate_splits(list(csv.DictReader(io.StringIO(manifest_bytes.decode("utf-8")))))\n        expected_mapping = {name: index for index, name in enumerate(CLASS_NAMES)}\n        if json.loads(contents["data/processed/class_to_idx.json"]) != expected_mapping:\n            raise ValueError("Backup class mapping does not match Day 1.")\n        for name, data in contents.items():\n            destination = (root / name).resolve()\n            if not destination.is_relative_to(root):\n                raise ValueError(f"Invalid backup path: {name}")\n            if destination.suffix.lower() not in {".csv", ".json", ".md", ".txt", ".jpg", ".png"}:\n                continue\n            destination.parent.mkdir(parents=True, exist_ok=True)\n            destination.write_bytes(data)\n    return validate_day1(root)\n\n\ndef validate_day1(root: Path) -> dict:\n    root = Path(root)\n    manifest = root / "data/processed/splits.csv"\n    actual = sha256_file(manifest)\n    summary = json.loads((root / "outputs/day1/data_summary.json").read_text(encoding="utf-8"))\n    if actual != EXPECTED_DAY1_MANIFEST_SHA256 or actual != summary["manifest_sha256"]:\n        raise ValueError("Day 1 manifest changed. Restore the verified Day 1 backup.")\n    with manifest.open(newline="", encoding="utf-8") as stream:\n        rows = list(csv.DictReader(stream))\n    validate_splits(rows)\n    mapping = json.loads((root / "data/processed/class_to_idx.json").read_text(encoding="utf-8"))\n    if mapping != {name: index for index, name in enumerate(CLASS_NAMES)}:\n        raise ValueError("Class mapping changed since Day 1.")\n    counts = {split: sum(row["split"] == split for row in rows) for split in ("train", "val", "test")}\n    if counts != summary["split_counts"]:\n        raise ValueError("Recorded split counts do not match the manifest.")\n    return {"manifest_sha256": actual, "class_to_idx": mapping, "split_counts": counts}\n\n\nclass SmallCNN(nn.Module):\n    """Three conv/BatchNorm/ReLU/pool blocks, followed by a small classifier."""\n    def __init__(self, num_classes=5):\n        super().__init__()\n        blocks = []\n        for incoming, outgoing in ((3, 16), (16, 32), (32, 64)):\n            blocks.extend([nn.Conv2d(incoming, outgoing, 3, padding=1, bias=False),\n                           nn.BatchNorm2d(outgoing), nn.ReLU(inplace=True), nn.MaxPool2d(2)])\n        self.features = nn.Sequential(*blocks)\n        self.pool = nn.AdaptiveAvgPool2d((4, 4))\n        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(64 * 4 * 4, 128),\n                                        nn.ReLU(inplace=True), nn.Dropout(0.3), nn.Linear(128, num_classes))\n\n    def forward(self, images):\n        return self.classifier(self.pool(self.features(images)))\n\n\ndef metrics_from_confusion(confusion: torch.Tensor) -> dict:\n    counts = confusion.to(dtype=torch.float64, device="cpu")\n    true_positive = counts.diag()\n    predicted = counts.sum(0)\n    actual = counts.sum(1)\n    precision = torch.where(predicted > 0, true_positive / predicted.clamp_min(1), 0.0)\n    recall = torch.where(actual > 0, true_positive / actual.clamp_min(1), 0.0)\n    denominator = precision + recall\n    f1 = torch.where(denominator > 0, 2 * precision * recall / denominator.clamp_min(1e-15), 0.0)\n    total = counts.sum().item()\n    return {"accuracy": float(true_positive.sum().item() / total) if total else 0.0,\n            "macro_f1": float(f1.mean().item()), "per_class_f1": f1.tolist(), "samples": int(total)}\n\n\ndef run_epoch(model, loader, device, optimizer=None, max_batches=None, log_every=20):\n    training = optimizer is not None\n    model.train(training)\n    confusion = torch.zeros((len(CLASS_NAMES), len(CLASS_NAMES)), dtype=torch.int64)\n    total_loss, seen = 0.0, 0\n    criterion = nn.CrossEntropyLoss()\n    started = time.perf_counter()\n    batches = islice(loader, max_batches) if max_batches is not None else loader\n    for index, (images, labels) in enumerate(batches):\n        images = images.to(device, non_blocking=True)\n        labels = labels.to(device, non_blocking=True)\n        if training:\n            optimizer.zero_grad(set_to_none=True)\n        with torch.set_grad_enabled(training):\n            logits = model(images)\n            loss = criterion(logits, labels)\n            if not torch.isfinite(loss):\n                raise FloatingPointError("Training/validation loss is not finite.")\n            if training:\n                loss.backward()\n                optimizer.step()\n        batch_size = labels.size(0)\n        total_loss += loss.item() * batch_size\n        seen += batch_size\n        codes = labels.detach().cpu() * len(CLASS_NAMES) + logits.detach().argmax(1).cpu()\n        confusion += torch.bincount(codes, minlength=len(CLASS_NAMES)**2).reshape(len(CLASS_NAMES), len(CLASS_NAMES))\n        if training and log_every and (index + 1) % log_every == 0:\n            print(f"    train batch {index + 1}/{len(loader)} | loss {total_loss / seen:.4f}", flush=True)\n    if seen == 0:\n        raise ValueError("Epoch received no images.")\n    metrics = metrics_from_confusion(confusion)\n    metrics.update(loss=total_loss / seen, seconds=time.perf_counter() - started)\n    return metrics\n\n\ndef state_fingerprint(state) -> str:\n    digest = hashlib.sha256()\n    for name, tensor in sorted(state.items()):\n        digest.update(name.encode())\n        digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())\n    return digest.hexdigest()\n\n\ndef train_one_run(root, directory, run_name, initial_state, config, data_info, augment, model_factory=None):\n    from torch.utils.tensorboard import SummaryWriter\n\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=False)\n    device = torch.device(config["device"])\n    loaders, _ = make_loaders(root, batch_size=config["batch_size"], num_workers=config["num_workers"],\n                              augment=augment, seed=config["seed"], crop_scale=config.get("crop_scale", (0.75, 1.0)))\n    seed_everything(config["seed"])\n    model = model_factory() if model_factory is not None else SmallCNN()\n    model.load_state_dict(initial_state)\n    model.to(device)\n    optimizer = torch.optim.AdamW((parameter for parameter in model.parameters() if parameter.requires_grad),\n                                  lr=config["learning_rate"], weight_decay=config["weight_decay"])\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["epochs"])\n    initial_hash = state_fingerprint(model.state_dict())\n    run_config = {**config, "run": run_name, "augmentation": augment,\n                  "manifest_sha256": data_info["manifest_sha256"], "initial_state_sha256": initial_hash,\n                  "class_to_idx": data_info["class_to_idx"], "selection_metric": "validation_macro_f1",\n                  "selection_tiebreak": "lower_validation_loss", "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad)}\n    write_json(directory / "config.json", run_config)\n    record_environment(directory / "environment.json")\n    history = []\n    best_score, best_loss, best_epoch = -1.0, math.inf, 0\n    started = time.perf_counter()\n    fields = ("epoch", "learning_rate", "train_loss", "train_accuracy", "train_macro_f1",\n              "val_loss", "val_accuracy", "val_macro_f1", "train_samples", "val_samples", "epoch_seconds")\n    writer = SummaryWriter(log_dir=str(directory / "tensorboard"))\n    try:\n        with (directory / "history.csv").open("w", newline="", encoding="utf-8") as stream:\n            history_writer = csv.DictWriter(stream, fieldnames=fields)\n            history_writer.writeheader()\n            for epoch in range(1, config["epochs"] + 1):\n                learning_rate = optimizer.param_groups[0]["lr"]\n                train = run_epoch(model, loaders["train"], device, optimizer,\n                                  config["max_train_batches"], log_every=config["log_every"])\n                validation = run_epoch(model, loaders["val"], device,\n                                       max_batches=config["max_val_batches"], log_every=0)\n                row = {"epoch": epoch, "learning_rate": learning_rate,\n                       "train_loss": train["loss"], "train_accuracy": train["accuracy"], "train_macro_f1": train["macro_f1"],\n                       "val_loss": validation["loss"], "val_accuracy": validation["accuracy"], "val_macro_f1": validation["macro_f1"],\n                       "train_samples": train["samples"], "val_samples": validation["samples"],\n                       "epoch_seconds": train["seconds"] + validation["seconds"]}\n                history.append(row)\n                history_writer.writerow(row)\n                stream.flush()\n                for stage, metrics in (("train", train), ("val", validation)):\n                    for metric in ("loss", "accuracy", "macro_f1"):\n                        writer.add_scalar(f"{metric}/{stage}", metrics[metric], epoch)\n                writer.add_scalar("learning_rate", learning_rate, epoch)\n                is_best = validation["macro_f1"] > best_score or (validation["macro_f1"] == best_score and validation["loss"] < best_loss)\n                if is_best:\n                    best_score, best_loss, best_epoch = validation["macro_f1"], validation["loss"], epoch\n                    checkpoint = {"model_state": {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()},\n                                  "model_name": config.get("model_name", "SmallCNN"), "num_classes": len(CLASS_NAMES), "epoch": epoch,\n                                  "val_macro_f1": best_score, "val_loss": best_loss, "config": run_config}\n                    pending = directory / "best.pt.tmp"\n                    torch.save(checkpoint, pending)\n                    pending.replace(directory / "best.pt")\n                print(f"{run_name} | epoch {epoch:02}/{config[\'epochs\']} | train loss {train[\'loss\']:.4f} "\n                      f"| val loss {validation[\'loss\']:.4f} | val accuracy {validation[\'accuracy\']:.1%} "\n                      f"| val macro-F1 {validation[\'macro_f1\']:.4f}{\' * best\' if is_best else \'\'}", flush=True)\n                writer.flush()\n                scheduler.step()\n    finally:\n        writer.close()\n    best_row = history[best_epoch - 1]\n    result = {"run": run_name, "mode": config["mode"], "augmentation": augment, "epochs": config["epochs"],\n              "best_epoch": best_epoch, "val_accuracy": best_row["val_accuracy"], "val_macro_f1": best_score,\n              "val_loss": best_loss, "training_seconds": time.perf_counter() - started,\n              "checkpoint": str((directory / "best.pt").resolve()), "manifest_sha256": data_info["manifest_sha256"],\n              "initial_state_sha256": initial_hash, "test_evaluated": False}\n    write_json(directory / "summary.json", result)\n    return result\n\n\ndef plot_comparison(experiment):\n    experiment = Path(experiment)\n    config = experiment / ".matplotlib"\n    config.mkdir(parents=True, exist_ok=True)\n    os.environ.setdefault("MPLCONFIGDIR", str(config.resolve()))\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n\n    metadata = json.loads((experiment / "experiment.json").read_text())\n    figure, axes = plt.subplots(1, 3, figsize=(16, 4.5), layout="constrained")\n    for run, color in (("cnn_baseline", "#2962d9"), ("cnn_augmented", "#249977")):\n        with (experiment / run / "history.csv").open(newline="", encoding="utf-8") as stream:\n            rows = list(csv.DictReader(stream))\n        epochs = [int(row["epoch"]) for row in rows]\n        label = "Baseline" if run == "cnn_baseline" else "Augmented"\n        axes[0].plot(epochs, [float(row["train_loss"]) for row in rows], "--", color=color, label=f"{label}: train")\n        axes[0].plot(epochs, [float(row["val_loss"]) for row in rows], color=color, label=f"{label}: val")\n        axes[1].plot(epochs, [float(row["val_accuracy"]) for row in rows], color=color, label=label)\n        axes[2].plot(epochs, [float(row["val_macro_f1"]) for row in rows], color=color, label=label)\n    for axis, title, ylabel in zip(axes, ("Training and validation loss", "Validation accuracy", "Validation macro-F1"),\n                                   ("Cross-entropy", "Accuracy", "Macro-F1")):\n        axis.set(title=title, xlabel="Epoch", ylabel=ylabel)\n        axis.legend(fontsize=8)\n        axis.grid(alpha=0.2)\n    axes[1].set_ylim(0, 1)\n    axes[2].set_ylim(0, 1)\n    figure.suptitle("CPU SMOKE CHECK - partial data; not project results" if metadata["mode"] == "smoke" else "Day 2 custom CNN comparison (validation)")\n    destination = experiment / "learning_curves.png"\n    figure.savefig(destination, dpi=150)\n    plt.close(figure)\n    return destination\n\n\ndef run_comparison(root: Path, epochs=12, batch_size=32, seed=42, num_workers=0, allow_cpu=False,\n                   smoke=False, output_dir=None, max_train_batches=None, max_val_batches=None):\n    root = Path(root).resolve()\n    if epochs < 1 or batch_size < 1 or num_workers < 0:\n        raise ValueError("Epochs/batch size must be positive and workers must be nonnegative.")\n    if not smoke and (max_train_batches is not None or max_val_batches is not None):\n        raise ValueError("Partial-data limits are only allowed with smoke=True.")\n    if smoke:\n        max_train_batches = 2 if max_train_batches is None else max_train_batches\n        max_val_batches = 2 if max_val_batches is None else max_val_batches\n        if min(max_train_batches, max_val_batches) < 1:\n            raise ValueError("Smoke batch limits must be positive.")\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    if device == "cpu" and not (allow_cpu or smoke):\n        raise RuntimeError("Enable a Colab GPU runtime first and rerun setup. For deliberate CPU training, set ALLOW_CPU=True.")\n    if device == "cpu":\n        torch.set_num_threads(min(4, os.cpu_count() or 1))\n    data_info = validate_day1(root)\n    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")\n    experiment = Path(output_dir) if output_dir else root / "outputs" / ("day2_smoke" if smoke else "day2") / stamp\n    experiment = experiment.resolve()\n    experiment.mkdir(parents=True, exist_ok=False)\n    config = {"mode": "smoke" if smoke else "full", "epochs": epochs, "batch_size": batch_size, "seed": seed,\n              "num_workers": num_workers, "learning_rate": 0.001, "weight_decay": 0.0001,\n              "optimizer": "AdamW",\n              "device": device, "max_train_batches": max_train_batches, "max_val_batches": max_val_batches,\n              "log_every": 0 if smoke else 20, "scheduler": "CosineAnnealingLR", "image_size": 224,\n              "gpu": torch.cuda.get_device_name(0) if device == "cuda" else None, "python": platform.python_version()}\n    write_json(experiment / "experiment.json", {**config, **data_info})\n    print(f"Experiment: {experiment}\\nMode: {config[\'mode\']} | device: {device} | epochs per run: {epochs}", flush=True)\n    seed_everything(seed)\n    initial_state = {name: tensor.detach().cpu().clone() for name, tensor in SmallCNN().state_dict().items()}\n    initial_hash = state_fingerprint(initial_state)\n    results = []\n    for name, augment in (("cnn_baseline", False), ("cnn_augmented", True)):\n        result = train_one_run(root, experiment / name, name, initial_state, config, data_info, augment)\n        if result["initial_state_sha256"] != initial_hash:\n            raise AssertionError("Paired runs did not start from identical weights.")\n        results.append(result)\n    with (experiment / "comparison.csv").open("w", newline="", encoding="utf-8") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(results[0]))\n        writer.writeheader()\n        writer.writerows(results)\n    plot_comparison(experiment)\n    winner = max(results, key=lambda row: (row["val_macro_f1"], -row["val_loss"]))\n    write_json(experiment / "comparison.json", {"mode": config["mode"], "runs": results,\n               "selected_by_validation": winner["run"], "test_evaluated": False,\n               "caveat": "One seed and one training budget. Augmentation is not guaranteed to improve validation results."})\n    print(f"Finished. Validation selection: {winner[\'run\']}. Test set was not evaluated.", flush=True)\n    return experiment\n\n\ndef package_day2(root: Path, experiment: Path) -> Path:\n    root, experiment = Path(root).resolve(), Path(experiment).resolve()\n    mode = json.loads((experiment / "experiment.json").read_text())["mode"]\n    destination = root / ("day2_smoke_backup.zip" if mode == "smoke" else "day2_backup.zip")\n    with zipfile.ZipFile(destination, "w", zipfile.ZIP_DEFLATED) as archive:\n        for name in ("flower_data.py", "flower_pipeline.py", "flower_training.py"):\n            archive.write(root / name, name)\n        for folder in (root / "data/processed", root / "outputs/day1", experiment):\n            for path in sorted(folder.rglob("*")):\n                if path.is_file() and ".matplotlib" not in path.parts:\n                    run_prefix = "outputs/day2_smoke" if mode == "smoke" else "outputs/day2"\n                    archive.write(path, path.relative_to(root) if folder != experiment else Path(run_prefix) / experiment.name / path.relative_to(experiment))\n        credits = root / "data/raw/flower_photos/LICENSE.txt"\n        if credits.is_file():\n            archive.write(credits, "docs/DATASET_IMAGE_CREDITS.txt")\n    return destination\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--root", type=Path, default=Path(__file__).resolve().parent)\n    parser.add_argument("--epochs", type=int, default=12)\n    parser.add_argument("--batch-size", type=int, default=32)\n    parser.add_argument("--num-workers", type=int, default=0)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--allow-cpu", action="store_true")\n    parser.add_argument("--smoke", action="store_true")\n    args = parser.parse_args()\n    run_comparison(args.root, epochs=args.epochs, batch_size=args.batch_size, seed=args.seed,\n                   num_workers=args.num_workers, allow_cpu=args.allow_cpu, smoke=args.smoke)\n', 'flower_transfer.py': '"""Day 3: pretrained ResNet18, a frozen-head stage, then full fine-tuning."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport os\nimport time\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path, PurePosixPath\n\nimport torch\nfrom torch import nn\nfrom torchvision.models import ResNet18_Weights, resnet18\n\nfrom flower_data import CLASS_NAMES, record_environment, sha256_file, write_json\nfrom flower_pipeline import seed_everything\nfrom flower_training import (EXPECTED_DAY1_MANIFEST_SHA256, restore_day1_backup,\n                             state_fingerprint, train_one_run, validate_day1)\n\nWEIGHTS = ResNet18_Weights.IMAGENET1K_V1\nDAY2_RUNS = {"cnn_baseline", "cnn_augmented"}\n\n\nclass FlowerResNet18(nn.Module):\n    """Frozen-head training keeps every backbone parameter and BN buffer fixed."""\n    def __init__(self, num_classes=5, pretrained=False, stage="head"):\n        super().__init__()\n        self.network = resnet18(weights=WEIGHTS if pretrained else None)\n        self.network.fc = nn.Linear(self.network.fc.in_features, num_classes)\n        self.set_stage(stage)\n\n    def set_stage(self, stage):\n        if stage not in {"head", "finetune"}:\n            raise ValueError("Stage must be head or finetune.")\n        self.stage = stage\n        for parameter in self.network.parameters():\n            parameter.requires_grad_(stage == "finetune")\n        for parameter in self.network.fc.parameters():\n            parameter.requires_grad_(True)\n        self.train(self.training)\n        return self\n\n    def train(self, mode=True):\n        super().train(mode)\n        if getattr(self, "stage", None) == "head":\n            self.network.eval()\n            self.network.fc.train(mode)\n        return self\n\n    def forward(self, images):\n        return self.network(images)\n\n\ndef backbone_fingerprint(state):\n    return state_fingerprint({name: value for name, value in state.items() if not name.startswith("network.fc.")})\n\n\ndef load_pretrained_state(root, seed=42):\n    """Use a project-local cache and torchvision\'s verified official weights."""\n    cache = Path(root).resolve() / "data/pretrained"\n    torch.hub.set_dir(str(cache))\n    seed_everything(seed)\n    model = FlowerResNet18(pretrained=True, stage="head")\n    state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}\n    path = cache / "checkpoints" / WEIGHTS.url.rsplit("/", 1)[1]\n    metadata = {"pretrained_weights": "ResNet18_Weights.IMAGENET1K_V1", "weights_url": WEIGHTS.url,\n                "pretrained_file_sha256": sha256_file(path), "initial_backbone_sha256": backbone_fingerprint(state)}\n    return state, metadata\n\n\ndef _validate_day2_comparison(comparison):\n    rows = comparison.get("runs", [])\n    if comparison.get("mode") != "full" or {row.get("run") for row in rows} != DAY2_RUNS or len(rows) != 2:\n        raise ValueError("Use the completed full Day 2 comparison backup.")\n    if comparison.get("test_evaluated") is not False:\n        raise ValueError("Day 2 comparison must leave the test set untouched.")\n    if len({row.get("initial_state_sha256") for row in rows}) != 1:\n        raise ValueError("Day 2 runs do not record the same initial weights.")\n    for row in rows:\n        if row.get("manifest_sha256") != EXPECTED_DAY1_MANIFEST_SHA256 or row.get("test_evaluated") is not False:\n            raise ValueError("Day 2 manifest/test-evaluation record is inconsistent.")\n        if row.get("epochs") != 12 or row.get("mode") != "full":\n            raise ValueError("Expected the verified 12-epoch Day 2 experiments.")\n    return rows\n\n\ndef restore_day2_backup(backup: Path, root: Path):\n    """Restore data and result artifacts only; archive Python is never executed."""\n    root = Path(root).resolve()\n    with zipfile.ZipFile(backup) as archive:\n        names = archive.namelist()\n        matches = [name for name in names if name.startswith("outputs/day2/") and name.endswith("/comparison.json")]\n        if len(matches) != 1:\n            raise ValueError("Expected exactly one full Day 2 comparison in the backup.")\n        comparison_name = matches[0]\n        prefix = comparison_name.rsplit("/", 1)[0] + "/"\n        comparison = json.loads(archive.read(comparison_name))\n        rows = _validate_day2_comparison(comparison)\n        selected = [info for info in archive.infolist() if not info.is_dir() and info.filename.startswith(prefix)]\n        if len({info.filename for info in selected}) != len(selected):\n            raise ValueError("Duplicate Day 2 result paths in archive.")\n        if any(info.file_size > 50 * 1024**2 for info in selected) or sum(info.file_size for info in selected) > 200 * 1024**2:\n            raise ValueError("Day 2 result artifacts are unexpectedly large.")\n        for info in selected:\n            parts = PurePosixPath(info.filename).parts\n            if ".." in parts or "\\\\" in info.filename or not (root / info.filename).resolve().is_relative_to(root / "outputs/day2"):\n                raise ValueError(f"Invalid result path: {info.filename}")\n        for row in rows:\n            for filename in ("best.pt", "history.csv", "config.json"):\n                if prefix + row["run"] + "/" + filename not in names:\n                    raise ValueError(f"Missing Day 2 artifact: {row[\'run\']}/{filename}")\n        # This also verifies and restores the unchanged original split files.\n        restore_day1_backup(backup, root)\n        for info in selected:\n            if Path(info.filename).suffix == ".py":\n                continue\n            target = root / info.filename\n            target.parent.mkdir(parents=True, exist_ok=True)\n            target.write_bytes(archive.read(info))\n        pointer = {"experiment": prefix.rstrip("/"), "comparison_sha256": hashlib.sha256(archive.read(comparison_name)).hexdigest(),\n                   "backup_sha256": sha256_file(Path(backup))}\n        write_json(root / "outputs/day2_import.json", pointer)\n    return load_day2_results(root)\n\n\ndef load_day2_results(root: Path):\n    root = Path(root).resolve()\n    pointer = json.loads((root / "outputs/day2_import.json").read_text(encoding="utf-8"))\n    experiment = (root / pointer["experiment"]).resolve()\n    if not experiment.is_relative_to(root / "outputs/day2"):\n        raise ValueError("Invalid restored Day 2 directory.")\n    comparison_path = experiment / "comparison.json"\n    if sha256_file(comparison_path) != pointer["comparison_sha256"]:\n        raise ValueError("The restored Day 2 comparison changed.")\n    comparison = json.loads(comparison_path.read_text(encoding="utf-8"))\n    _validate_day2_comparison(comparison)\n    validate_day1(root)\n    return experiment, comparison\n\n\ndef _comparison_row(row, model_name, stage_epochs, cumulative_epochs, cumulative_seconds):\n    return {"run": row["run"], "model": model_name, "best_epoch_in_stage": row["best_epoch"],\n            "stage_epoch_budget": stage_epochs, "cumulative_epoch_budget": cumulative_epochs,\n            "val_accuracy": row["val_accuracy"], "val_macro_f1": row["val_macro_f1"], "val_loss": row["val_loss"],\n            "stage_seconds": row["training_seconds"], "cumulative_seconds": cumulative_seconds,\n            "manifest_sha256": row["manifest_sha256"], "test_evaluated": False}\n\n\ndef plot_transfer_comparison(experiment):\n    import csv\n    experiment = Path(experiment)\n    os.environ.setdefault("MPLCONFIGDIR", str(experiment / ".matplotlib"))\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n\n    metadata = json.loads((experiment / "experiment.json").read_text())\n    figure, axes = plt.subplots(1, 3, figsize=(16, 4.5), layout="constrained")\n    for name, label, color in (("resnet18_head", "Frozen head", "#2962d9"),\n                               ("resnet18_finetuned", "Fine-tuned", "#249977")):\n        with (experiment / name / "history.csv").open(newline="", encoding="utf-8") as stream:\n            rows = list(csv.DictReader(stream))\n        epochs = [int(row["epoch"]) for row in rows]\n        axes[0].plot(epochs, [float(row["train_loss"]) for row in rows], "--", color=color, label=f"{label}: train")\n        axes[0].plot(epochs, [float(row["val_loss"]) for row in rows], color=color, label=f"{label}: val")\n        axes[1].plot(epochs, [float(row["val_accuracy"]) for row in rows], color=color, label=label)\n        axes[2].plot(epochs, [float(row["val_macro_f1"]) for row in rows], color=color, label=label)\n    for axis, title, ylabel in zip(axes, ("Training/validation loss", "Validation accuracy", "Validation macro-F1"),\n                                   ("Cross-entropy", "Accuracy", "Macro-F1")):\n        axis.set(title=title, xlabel="Epoch within stage", ylabel=ylabel)\n        axis.grid(alpha=0.2)\n        axis.legend(fontsize=8)\n    axes[1].set_ylim(0, 1)\n    axes[2].set_ylim(0, 1)\n    figure.suptitle("CPU SMOKE CHECK - partial data; not project results" if metadata["mode"] == "smoke"\n                    else "Day 3 ResNet18 stages (fine-tuning starts from the best head checkpoint)")\n    destination = experiment / "learning_curves.png"\n    figure.savefig(destination, dpi=150)\n    plt.close(figure)\n    return destination\n\n\ndef run_transfer_comparison(root, head_epochs=6, fine_epochs=6, batch_size=32, seed=42,\n                            num_workers=0, allow_cpu=False, smoke=False, output_dir=None):\n    import csv\n    root = Path(root).resolve()\n    if min(head_epochs, fine_epochs, batch_size) < 1 or num_workers < 0:\n        raise ValueError("Epochs/batch size must be positive and workers nonnegative.")\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    if device == "cpu" and not (allow_cpu or smoke):\n        raise RuntimeError("Select a Colab GPU runtime and rerun setup, or deliberately set ALLOW_CPU=True.")\n    if device == "cpu":\n        torch.set_num_threads(min(4, os.cpu_count() or 1))\n    source_experiment, day2 = load_day2_results(root)\n    data_info = validate_day1(root)\n    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")\n    experiment = Path(output_dir) if output_dir else root / "outputs" / ("day3_smoke" if smoke else "day3") / stamp\n    experiment = experiment.resolve()\n    experiment.mkdir(parents=True, exist_ok=False)\n    print(f"Experiment: {experiment}\\nLoading official ImageNet-pretrained ResNet18 weights...", flush=True)\n    initial_state, provenance = load_pretrained_state(root, seed)\n    common = {"mode": "smoke" if smoke else "full", "batch_size": batch_size, "seed": seed,\n              "num_workers": num_workers, "weight_decay": 0.0001, "optimizer": "AdamW", "device": device,\n              "max_train_batches": 1 if smoke else None, "max_val_batches": 1 if smoke else None,\n              "log_every": 0 if smoke else 20, "scheduler": "CosineAnnealingLR", "image_size": 224,\n              "model_name": "FlowerResNet18", "gpu": torch.cuda.get_device_name(0) if device == "cuda" else None,\n              **provenance}\n    write_json(experiment / "experiment.json", {**common, **data_info, "head_epochs": head_epochs, "fine_epochs": fine_epochs,\n               "day2_source": str(source_experiment.relative_to(root)), "comparison_budget_note":\n               "Frozen head uses its stage budget. Fine-tuning adds to that budget and starts from the best head checkpoint."})\n    head_config = {**common, "epochs": head_epochs, "learning_rate": 0.001, "stage": "head", "backbone_frozen": True}\n    head = train_one_run(root, experiment / "resnet18_head", "resnet18_head", initial_state,\n                         head_config, data_info, augment=True, model_factory=lambda: FlowerResNet18(stage="head"))\n    head_checkpoint = torch.load(experiment / "resnet18_head/best.pt", map_location="cpu", weights_only=True)\n    head_state = head_checkpoint["model_state"]\n    if backbone_fingerprint(head_state) != provenance["initial_backbone_sha256"]:\n        raise AssertionError("Frozen-head training changed backbone weights or BatchNorm buffers.")\n    parent_hash = state_fingerprint(head_state)\n    fine_config = {**common, "epochs": fine_epochs, "learning_rate": 0.0001, "stage": "finetune", "backbone_frozen": False,\n                   "parent_run": "resnet18_head", "parent_best_epoch": head_checkpoint["epoch"],\n                   "parent_model_sha256": parent_hash, "parent_checkpoint_sha256": sha256_file(experiment / "resnet18_head/best.pt")}\n    fine = train_one_run(root, experiment / "resnet18_finetuned", "resnet18_finetuned", head_state,\n                         fine_config, data_info, augment=True, model_factory=lambda: FlowerResNet18(stage="finetune"))\n    if fine["initial_state_sha256"] != parent_hash:\n        raise AssertionError("Fine-tuning did not start from the best frozen-head checkpoint.")\n    rows = [_comparison_row(row, "SmallCNN", row["epochs"], row["epochs"], row["training_seconds"]) for row in day2["runs"]]\n    rows.extend([_comparison_row(head, "ResNet18", head_epochs, head_epochs, head["training_seconds"]),\n                 _comparison_row(fine, "ResNet18", fine_epochs, head_epochs + fine_epochs,\n                                 head["training_seconds"] + fine["training_seconds"])])\n    if smoke:\n        for row in rows:\n            row["verification_only"] = True\n    with (experiment / "comparison.csv").open("w", newline="", encoding="utf-8") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n    winner = max(rows, key=lambda row: (row["val_macro_f1"], -row["val_loss"]))\n    write_json(experiment / "comparison.json", {"mode": common["mode"], "runs": rows,\n               "selected_by_validation": None if smoke else winner["run"], "test_evaluated": False,\n               "head_backbone_unchanged": True, "fine_tune_parent_verified": True,\n               "day2_source": str(source_experiment.relative_to(root)),\n               "caveats": ["One seed; different model capacities and pretraining histories.",\n                           "Fine-tuned cumulative time includes the frozen-head stage; head-only has fewer epochs."]})\n    plot_transfer_comparison(experiment)\n    print("Both stages finished. Test set was not evaluated.", flush=True)\n    return experiment\n\n\ndef package_day3(root, experiment):\n    root, experiment = Path(root).resolve(), Path(experiment).resolve()\n    mode = json.loads((experiment / "experiment.json").read_text())["mode"]\n    day2_experiment, _ = load_day2_results(root)\n    destination = root / ("day3_smoke_backup.zip" if mode == "smoke" else "day3_backup.zip")\n    with zipfile.ZipFile(destination, "w", zipfile.ZIP_DEFLATED) as archive:\n        for name in ("flower_data.py", "flower_pipeline.py", "flower_training.py", "flower_transfer.py", "outputs/day2_import.json"):\n            archive.write(root / name, name)\n        for directory in (root / "data/processed", root / "outputs/day1", day2_experiment, experiment):\n            for path in sorted(directory.rglob("*")):\n                if path.is_file() and ".matplotlib" not in path.parts:\n                    archive.write(path, path.relative_to(root))\n        credits = root / "data/raw/flower_photos/LICENSE.txt"\n        if credits.is_file():\n            archive.write(credits, "docs/DATASET_IMAGE_CREDITS.txt")\n    return destination\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--root", type=Path, default=Path(__file__).resolve().parent)\n    parser.add_argument("--head-epochs", type=int, default=6)\n    parser.add_argument("--fine-epochs", type=int, default=6)\n    parser.add_argument("--batch-size", type=int, default=32)\n    parser.add_argument("--allow-cpu", action="store_true")\n    parser.add_argument("--smoke", action="store_true")\n    args = parser.parse_args()\n    run_transfer_comparison(args.root, head_epochs=args.head_epochs, fine_epochs=args.fine_epochs,\n                            batch_size=args.batch_size, allow_cpu=args.allow_cpu, smoke=args.smoke)\n', 'flower_evaluation.py': '"""Day 4: validation metrics, confusion matrices, and recorded Grad-CAM review."""\n\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport html\nimport base64\nimport json\nimport os\nimport random\nimport time\nimport zipfile\nfrom collections import defaultdict\nfrom datetime import datetime, timezone\nfrom pathlib import Path, PurePosixPath\n\nimport numpy as np\nimport torch\nfrom PIL import Image, ImageDraw, ImageFont\nfrom torch import nn\nfrom torch.nn import functional as F\n\nfrom flower_data import CLASS_NAMES, record_environment, sha256_file, write_json\nfrom flower_pipeline import IMAGENET_MEAN, IMAGENET_STD, make_loaders\nfrom flower_training import EXPECTED_DAY1_MANIFEST_SHA256, SmallCNN, state_fingerprint, validate_day1\nfrom flower_transfer import FlowerResNet18, load_day2_results, restore_day2_backup\n\nMODEL_NAMES = ("cnn_baseline", "cnn_augmented", "resnet18_head", "resnet18_finetuned")\n\n\ndef restore_day3_backup(backup, root):\n    root = Path(root).resolve()\n    with zipfile.ZipFile(backup) as archive:\n        matches = [name for name in archive.namelist() if name.startswith("outputs/day3/") and name.endswith("/comparison.json")]\n        if len(matches) != 1:\n            raise ValueError("Select the completed full Day 3 backup ZIP.")\n        comparison_name = matches[0]\n        prefix = comparison_name.rsplit("/", 1)[0] + "/"\n        comparison = json.loads(archive.read(comparison_name))\n        if comparison.get("mode") != "full" or {row["run"] for row in comparison["runs"]} != set(MODEL_NAMES):\n            raise ValueError("Day 3 comparison must contain all four full model results.")\n        if comparison.get("test_evaluated") is not False or comparison.get("selected_by_validation") not in MODEL_NAMES:\n            raise ValueError("Invalid Day 3 selection or test-evaluation record.")\n        if any(row["manifest_sha256"] != EXPECTED_DAY1_MANIFEST_SHA256 for row in comparison["runs"]):\n            raise ValueError("Day 3 splits do not match the accepted original splits.")\n        selected = [info for info in archive.infolist() if not info.is_dir() and info.filename.startswith(prefix)]\n        if len({info.filename for info in selected}) != len(selected):\n            raise ValueError("Duplicate Day 3 archive paths.")\n        if any(info.file_size > 128 * 1024**2 for info in selected) or sum(info.file_size for info in selected) > 512 * 1024**2:\n            raise ValueError("Unexpectedly large Day 3 artifacts.")\n        for info in selected:\n            target = (root / info.filename).resolve()\n            if ".." in PurePosixPath(info.filename).parts or "\\\\" in info.filename or not target.is_relative_to(root / "outputs/day3"):\n                raise ValueError(f"Invalid archive path: {info.filename}")\n        for run in ("resnet18_head", "resnet18_finetuned"):\n            if prefix + run + "/best.pt" not in archive.namelist():\n                raise ValueError(f"Missing checkpoint for {run}.")\n        restore_day2_backup(backup, root)\n        for info in selected:\n            if Path(info.filename).suffix == ".py":\n                continue\n            target = root / info.filename\n            target.parent.mkdir(parents=True, exist_ok=True)\n            target.write_bytes(archive.read(info))\n        write_json(root / "outputs/day3_import.json", {\n            "experiment": prefix.rstrip("/"),\n            "comparison_sha256": hashlib.sha256(archive.read(comparison_name)).hexdigest(),\n            "backup_sha256": sha256_file(Path(backup)),\n        })\n    return load_day3_results(root)\n\n\ndef load_day3_results(root):\n    root = Path(root).resolve()\n    pointer = json.loads((root / "outputs/day3_import.json").read_text(encoding="utf-8"))\n    directory = (root / pointer["experiment"]).resolve()\n    if not directory.is_relative_to(root / "outputs/day3"):\n        raise ValueError("Invalid Day 3 result directory.")\n    comparison_file = directory / "comparison.json"\n    if sha256_file(comparison_file) != pointer["comparison_sha256"]:\n        raise ValueError("The restored Day 3 comparison changed.")\n    comparison = json.loads(comparison_file.read_text(encoding="utf-8"))\n    if comparison.get("mode") != "full" or comparison.get("test_evaluated") is not False:\n        raise ValueError("Use full Day 3 results with the test set untouched.")\n    day2_directory, _ = load_day2_results(root)\n    if (root / comparison["day2_source"]).resolve() != day2_directory:\n        raise ValueError("Day 2 and Day 3 source comparisons are inconsistent.")\n    validate_day1(root)\n    checkpoints = {name: (day2_directory if name.startswith("cnn_") else directory) / name / "best.pt" for name in MODEL_NAMES}\n    return directory, comparison, checkpoints\n\n\ndef load_model(checkpoint_path, device):\n    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n    config = checkpoint["config"]\n    if config["manifest_sha256"] != EXPECTED_DAY1_MANIFEST_SHA256 or config["class_to_idx"] != dict(zip(CLASS_NAMES, range(5))):\n        raise ValueError("Checkpoint split hash or label mapping changed.")\n    if checkpoint["model_name"] == "SmallCNN":\n        model = SmallCNN()\n    elif checkpoint["model_name"] == "FlowerResNet18":\n        model = FlowerResNet18(stage=config["stage"])\n    else:\n        raise ValueError("Unsupported model architecture.")\n    model.load_state_dict(checkpoint["model_state"])\n    model.to(device).eval()\n    for parameter in model.parameters():\n        parameter.requires_grad_(False)\n    return model, checkpoint\n\n\ndef classification_report_from_matrix(matrix, names):\n    matrix = np.asarray(matrix, dtype=np.int64)\n    if matrix.shape != (len(names), len(names)) or np.any(matrix < 0):\n        raise ValueError("Confusion matrix has invalid shape or counts.")\n    support = matrix.sum(1)\n    predicted = matrix.sum(0)\n    diagonal = np.diag(matrix).astype(float)\n    precision = np.divide(diagonal, predicted, out=np.zeros(len(names)), where=predicted > 0)\n    recall = np.divide(diagonal, support, out=np.zeros(len(names)), where=support > 0)\n    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros(len(names)), where=(precision + recall) > 0)\n    total = int(support.sum())\n    report = {name: {"precision": float(precision[i]), "recall": float(recall[i]), "f1-score": float(f1[i]),\n                     "support": int(support[i])} for i, name in enumerate(names)}\n    report["accuracy"] = float(diagonal.sum() / total) if total else 0.0\n    report["macro avg"] = {"precision": float(precision.mean()), "recall": float(recall.mean()),\n                           "f1-score": float(f1.mean()), "support": total}\n    report["weighted avg"] = {metric: float(np.dot(values, support) / total) if total else 0.0\n                              for metric, values in (("precision", precision), ("recall", recall), ("f1-score", f1))}\n    report["weighted avg"]["support"] = total\n    return report\n\n\ndef predict_split(model, loader, device, split="val", max_batches=None):\n    if split not in {"val", "test"}:\n        raise ValueError("Prediction reports support validation or test data only.")\n    records, loss_sum = [], 0.0\n    model.eval()\n    with torch.inference_mode():\n        for batch_index, (images, labels) in enumerate(loader):\n            if max_batches is not None and batch_index >= max_batches:\n                break\n            logits = model(images.to(device, non_blocking=True))\n            loss_sum += F.cross_entropy(logits, labels.to(device), reduction="sum").item()\n            probabilities = logits.softmax(1).cpu()\n            predictions = probabilities.argmax(1)\n            for offset in range(len(labels)):\n                index = len(records)\n                source = loader.dataset.rows[index]\n                truth, prediction = int(labels[offset]), int(predictions[offset])\n                if int(source["label"]) != truth or source["split"] != split:\n                    raise ValueError("Prediction record order or split is inconsistent.")\n                top = probabilities[offset].topk(2).values\n                record = {"index": index, "split": split, "relative_path": source["relative_path"],\n                          "true_label": truth, "true_class": CLASS_NAMES[truth], "predicted_label": prediction,\n                          "predicted_class": CLASS_NAMES[prediction], "correct": truth == prediction,\n                          "confidence": float(top[0]), "top2_margin": float(top[0] - top[1])}\n                record.update({f"prob_{name}": float(probabilities[offset, i]) for i, name in enumerate(CLASS_NAMES)})\n                records.append(record)\n            if (batch_index + 1) % 5 == 0:\n                print(f"  {split} batch {batch_index + 1}/{len(loader)}", flush=True)\n    if not records:\n        raise ValueError("No predictions were produced.")\n    return records, loss_sum / len(records)\n\n\ndef predict_validation(model, loader, device, max_batches=None):\n    return predict_split(model, loader, device, split="val", max_batches=max_batches)\n\n\ndef configure_matplotlib(directory):\n    config = Path(directory) / ".matplotlib"\n    config.mkdir(parents=True, exist_ok=True)\n    os.environ.setdefault("MPLCONFIGDIR", str(config.resolve()))\n    import matplotlib\n    matplotlib.use("Agg")\n\n\ndef save_confusion_figure(matrix, destination, title):\n    import matplotlib.pyplot as plt\n    figure, axes = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")\n    support = matrix.sum(1, keepdims=True)\n    normalized = np.divide(matrix, support, out=np.zeros_like(matrix, dtype=float), where=support > 0)\n    for axis, values, label in zip(axes, (matrix, normalized), ("Counts", "Fraction within true class")):\n        image = axis.imshow(values, cmap="Blues", vmin=0, vmax=None if label == "Counts" else 1)\n        figure.colorbar(image, ax=axis, fraction=0.046)\n        axis.set(xticks=range(5), yticks=range(5), xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,\n                 xlabel="Predicted class", ylabel="True class", title=label)\n        axis.tick_params(axis="x", labelrotation=30)\n        threshold = float(values.max()) / 2\n        for i in range(5):\n            for j in range(5):\n                text = str(int(values[i, j])) if label == "Counts" else f"{values[i, j]:.2f}"\n                axis.text(j, i, text, ha="center", va="center", color="white" if values[i, j] > threshold else "black")\n    figure.suptitle(title)\n    figure.savefig(destination, dpi=140)\n    plt.close(figure)\n\n\ndef select_review_cases(records, per_outcome=20, seed=42):\n    if per_outcome < 1:\n        raise ValueError("Review count must be positive.")\n    selected = []\n    for outcome in (True, False):\n        groups = defaultdict(list)\n        for record in records:\n            if bool(record["correct"]) == outcome:\n                groups[int(record["true_label"])].append(record)\n        for label, group in groups.items():\n            if outcome:\n                group.sort(key=lambda row: row["relative_path"])\n                random.Random(f"{seed}:{label}:correct").shuffle(group)\n            else:\n                group.sort(key=lambda row: (-row["confidence"], row["relative_path"]))\n        chosen = []\n        while len(chosen) < per_outcome and any(groups.values()):\n            for label in sorted(groups):\n                if groups[label] and len(chosen) < per_outcome:\n                    chosen.append(dict(groups[label].pop(0)))\n        selected.extend(chosen)\n    return selected\n\n\ndef save_per_class_f1_figure(reports, destination):\n    import matplotlib.pyplot as plt\n    figure, axis = plt.subplots(figsize=(10, 5), layout="constrained")\n    positions = np.arange(5)\n    for index, name in enumerate(MODEL_NAMES):\n        axis.bar(positions + (index - 1.5) * 0.18, [reports[name][label]["f1-score"] for label in CLASS_NAMES],\n                 width=0.18, label=name)\n    axis.set(xticks=positions, xticklabels=CLASS_NAMES, ylim=(0, 1.05), ylabel="F1 score", title="Validation per-class F1")\n    axis.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))\n    figure.savefig(destination, dpi=150)\n    plt.close(figure)\n\n\ndef compute_gradcam(model, image_tensor, target_layer, target_classes):\n    """Class-logit Grad-CAM. Input gradients support a completely frozen backbone."""\n    if image_tensor.ndim != 4 or image_tensor.size(0) != 1:\n        raise ValueError("Grad-CAM expects one image at a time.")\n    targets = list(dict.fromkeys(int(label) for label in target_classes))\n    if not targets:\n        raise ValueError("At least one target class is required.")\n    captured = {}\n    def capture(module, inputs, output):\n        captured["activation"] = output\n    handle = target_layer.register_forward_hook(capture)\n    was_training = model.training\n    model.eval()\n    maps, flat = {}, {}\n    try:\n        with torch.inference_mode(False), torch.enable_grad():\n            image = image_tensor.detach().clone().requires_grad_(True)\n            logits = model(image)\n            activation = captured["activation"]\n            for position, label in enumerate(targets):\n                if label < 0 or label >= logits.size(1):\n                    raise ValueError("Grad-CAM target class is out of range.")\n                gradient = torch.autograd.grad(logits[0, label], activation, retain_graph=position < len(targets) - 1)[0]\n                weights = gradient.mean(dim=(2, 3), keepdim=True)\n                cam = torch.relu((weights * activation).sum(1, keepdim=True))\n                cam = F.interpolate(cam, size=image.shape[-2:], mode="bilinear", align_corners=False)[0, 0]\n                spread = cam.max() - cam.min()\n                flat[label] = bool(spread.detach().item() < 1e-12)\n                cam = torch.zeros_like(cam) if flat[label] else (cam - cam.min()) / spread\n                maps[label] = cam.detach().cpu().numpy()\n            scores = logits.detach().cpu()\n    finally:\n        handle.remove()\n        captured.clear()\n        model.train(was_training)\n    return maps, flat, scores\n\n\ndef target_layer_for(model):\n    if isinstance(model, FlowerResNet18):\n        return model.network.layer4[-1]\n    if isinstance(model, SmallCNN):\n        return next(module for module in reversed(list(model.features)) if isinstance(module, nn.Conv2d))\n    raise ValueError("Unsupported Grad-CAM model.")\n\n\ndef save_case_panel(tensor, maps, record, destination):\n    import matplotlib\n    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)\n    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)\n    rgb = (tensor.detach().cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()\n    truth, prediction = record["true_label"], record["predicted_label"]\n    source = Image.fromarray(np.uint8(np.round(rgb * 255)))\n    overlays = []\n    for label in (prediction, truth):\n        color = matplotlib.colormaps["inferno"](maps[label])[..., :3]\n        overlays.append(Image.fromarray(np.uint8(np.round((0.6 * rgb + 0.4 * color) * 255))))\n    canvas = Image.new("RGB", (3 * 224, 292), "#f5f7fb")\n    draw = ImageDraw.Draw(canvas)\n    font = ImageFont.load_default(size=13)\n    headings = ("Model input crop", f"Predicted: {CLASS_NAMES[prediction]}", f"True: {CLASS_NAMES[truth]}")\n    for index, (image, heading) in enumerate(zip((source, *overlays), headings)):\n        draw.text((index * 224 + 6, 7), heading, fill="#17253c", font=font)\n        canvas.paste(image, (index * 224, 30))\n    draw.text((6, 259), f"{\'CORRECT\' if record[\'correct\'] else \'INCORRECT\'} | softmax score {record[\'confidence\']:.1%}", fill="#17253c", font=font)\n    draw.text((6, 277), record["relative_path"], fill="#17253c", font=ImageFont.load_default(size=10))\n    canvas.save(destination)\n\n\ndef write_csv(path, records):\n    if not records:\n        return\n    with Path(path).open("w", newline="", encoding="utf-8") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(records[0]))\n        writer.writeheader()\n        writer.writerows(records)\n\n\ndef review_progress(directory):\n    cases = json.loads((Path(directory) / "review_cases.json").read_text(encoding="utf-8"))\n    human = [case for case in cases if case.get("reviewed") and case.get("reviewer") == "human"]\n    return {"total": len(cases), "reviewed": len(human), "remaining": len(cases) - len(human),\n            "correct_reviewed": sum(bool(case["correct"]) for case in human),\n            "incorrect_reviewed": sum(not case["correct"] for case in human)}\n\n\ndef save_case_review(directory, case_id, attention, pattern, comment="", reviewer="human"):\n    if attention not in {"flower", "background", "mixed", "unclear"}:\n        raise ValueError("Choose a valid attention category.")\n    if not pattern or not pattern.strip():\n        raise ValueError("Choose an observation/pattern.")\n    directory = Path(directory)\n    cases = json.loads((directory / "review_cases.json").read_text(encoding="utf-8"))\n    case = next((case for case in cases if case["case_id"] == case_id), None)\n    if case is None:\n        raise ValueError("Unknown review case.")\n    case.update(reviewed=True, reviewer=reviewer, attention=attention, pattern=pattern, comment=comment.strip(),\n                reviewed_at_utc=datetime.now(timezone.utc).isoformat())\n    write_json(directory / "review_cases.json", cases)\n    write_csv(directory / "review_cases.csv", cases)\n    return review_progress(directory)\n\n\ndef run_day4_analysis(root, batch_size=32, num_workers=0, per_outcome=20, seed=42, smoke=False):\n    root = Path(root).resolve()\n    data_info = validate_day1(root)\n    source_directory, day3, checkpoints = load_day3_results(root)\n    selected_name = day3["selected_by_validation"]\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    if device.type == "cpu":\n        torch.set_num_threads(min(4, os.cpu_count() or 1))\n    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")\n    directory = root / "outputs" / ("day4_smoke" if smoke else "day4") / stamp\n    directory.mkdir(parents=True, exist_ok=False)\n    configure_matplotlib(directory)\n    config = {"mode": "smoke" if smoke else "full", "split": "val", "test_evaluated": False, "seed": seed,\n              "batch_size": batch_size, "num_workers": num_workers, "selected_model": selected_name,\n              "requested_per_outcome": per_outcome, "manifest_sha256": data_info["manifest_sha256"],\n              "source_day3": str(source_directory.relative_to(root)), "device": str(device),\n              "review_selection": "Round-robin true classes; seeded correct samples; highest-confidence errors within class."}\n    write_json(directory / "analysis.json", config)\n    record_environment(directory / "environment.json")\n    loaders, _ = make_loaders(root, batch_size=batch_size, num_workers=num_workers, augment=False, seed=seed)\n    loader = loaders["val"]\n    summaries, all_reports = [], {}\n    selected_records = None\n    for name in MODEL_NAMES:\n        print(f"Evaluating {name} on validation data...", flush=True)\n        model, checkpoint = load_model(checkpoints[name], device)\n        started = time.perf_counter()\n        records, loss = predict_validation(model, loader, device, max_batches=2 if smoke else None)\n        seconds = time.perf_counter() - started\n        matrix = np.zeros((5, 5), dtype=np.int64)\n        for record in records:\n            matrix[record["true_label"], record["predicted_label"]] += 1\n        report = classification_report_from_matrix(matrix, CLASS_NAMES)\n        result_dir = directory / name\n        result_dir.mkdir()\n        write_json(result_dir / "classification_report.json", report)\n        write_json(result_dir / "confusion_matrix.json", {"class_names": list(CLASS_NAMES), "counts": matrix.tolist()})\n        write_csv(result_dir / "predictions.csv", records)\n        per_class = [{"class": label, **report[label]} for label in CLASS_NAMES]\n        write_csv(result_dir / "per_class_metrics.csv", per_class)\n        save_confusion_figure(matrix, result_dir / "confusion_matrix.png", f"{name}: validation")\n        summary = {"run": name, "accuracy": report["accuracy"], "macro_f1": report["macro avg"]["f1-score"],\n                   "weighted_f1": report["weighted avg"]["f1-score"], "loss": loss, "samples": len(records),\n                   "correct": int(np.trace(matrix)), "incorrect": int(matrix.sum() - np.trace(matrix)),\n                   "inference_seconds": seconds, "checkpoint_sha256": sha256_file(checkpoints[name])}\n        summaries.append(summary)\n        all_reports[name] = report\n        if name == selected_name:\n            selected_records = records\n            selected_matrix = matrix\n        print(f"  accuracy {summary[\'accuracy\']:.2%} | macro-F1 {summary[\'macro_f1\']:.4f} | errors {summary[\'incorrect\']}", flush=True)\n        del model, checkpoint\n    write_csv(directory / "model_metrics.csv", summaries)\n    write_json(directory / "classification_reports.json", all_reports)\n    save_per_class_f1_figure(all_reports, directory / "per_class_f1.png")\n    selected_report = all_reports[selected_name]\n    pairs = sorted([{"true_class": CLASS_NAMES[i], "predicted_class": CLASS_NAMES[j], "count": int(selected_matrix[i, j])}\n                    for i in range(5) for j in range(5) if i != j and selected_matrix[i, j] > 0],\n                   key=lambda row: (-row["count"], row["true_class"], row["predicted_class"]))\n    write_json(directory / "failure_patterns.json", {"selected_model": selected_name, "confusion_pairs": pairs,\n               "lowest_f1_class": min(CLASS_NAMES, key=lambda label: selected_report[label]["f1-score"]),\n               "note": "Confusion counts are measured; background bias and label problems require visual review."})\n    cases = select_review_cases(selected_records, per_outcome, seed)\n    model, _ = load_model(checkpoints[selected_name], device)\n    before = state_fingerprint(model.state_dict())\n    case_dir = directory / "gradcam"\n    case_dir.mkdir()\n    gallery = ["<!doctype html><meta charset=\'utf-8\'><title>Day 4 validation Grad-CAM review</title>",\n               "<h1>Validation Grad-CAM review</h1><p>Each panel uses the exact model input crop. Heatmaps are normalized independently. They are class-sensitivity maps, not segmentation or proof of causation. Softmax scores are uncalibrated.</p>"]\n    for position, case in enumerate(cases, 1):\n        tensor, label = loader.dataset[case["index"]]\n        maps, flat, logits = compute_gradcam(model, tensor.unsqueeze(0).to(device), target_layer_for(model),\n                                            [case["predicted_label"], case["true_label"]])\n        if int(logits.argmax(1)) != case["predicted_label"]:\n            raise ValueError("Grad-CAM prediction differs from the saved validation prediction.")\n        case_id = f"val_{case[\'index\']:04d}"\n        image_path = case_dir / f"{case_id}.png"\n        save_case_panel(tensor, maps, case, image_path)\n        np.savez_compressed(case_dir / f"{case_id}.npz", predicted=maps[case["predicted_label"]], truth=maps[case["true_label"]])\n        case.update(case_id=case_id, panel=str(image_path.relative_to(directory)),\n                    predicted_heatmap_flat=flat[case["predicted_label"]], true_heatmap_flat=flat[case["true_label"]],\n                    reviewed=False, reviewer="", attention="", pattern="", comment="", reviewed_at_utc="")\n        encoded = base64.b64encode(image_path.read_bytes()).decode()\n        gallery.append(f"<h2>{position}. {html.escape(case_id)}</h2><img style=\'max-width:100%\' src=\'data:image/png;base64,{encoded}\'><p>{html.escape(case[\'relative_path\'])}</p>")\n        print(f"Grad-CAM {position}/{len(cases)}: {case_id} ({\'correct\' if case[\'correct\'] else \'incorrect\'})", flush=True)\n    if state_fingerprint(model.state_dict()) != before:\n        raise AssertionError("Evaluation/Grad-CAM changed model parameters or buffers.")\n    write_json(directory / "review_cases.json", cases)\n    write_csv(directory / "review_cases.csv", cases)\n    (directory / "gradcam_gallery.html").write_text("\\n".join(gallery), encoding="utf-8")\n    correct_cases = sum(case["correct"] for case in cases)\n    wrong_cases = len(cases) - correct_cases\n    result = {**config, "model_metrics": summaries, "correct_cases": correct_cases, "incorrect_cases": wrong_cases,\n              "review_target_met": correct_cases >= 20 and wrong_cases >= 20, "manual_review_complete": False,\n              "model_state_unchanged": True, "gradcam_target": "predicted class plus true class for errors"}\n    write_json(directory / "summary.json", result)\n    lines = ["# Day 4 validation evaluation", "", f"Selected model: {selected_name}. Test set was not evaluated.", "",\n             "| Model | Accuracy | Macro-F1 | Errors |", "|---|---:|---:|---:|"]\n    lines += [f"| {row[\'run\']} | {row[\'accuracy\']:.2%} | {row[\'macro_f1\']:.4f} | {row[\'incorrect\']} |" for row in summaries]\n    lines += ["", f"## Per-class report: {selected_name}", "",\n              "| Class | Precision | Recall | F1 | Support |", "|---|---:|---:|---:|---:|"]\n    lines += [f"| {label} | {selected_report[label][\'precision\']:.4f} | {selected_report[label][\'recall\']:.4f} | "\n              f"{selected_report[label][\'f1-score\']:.4f} | {selected_report[label][\'support\']} |" for label in CLASS_NAMES]\n    lines += ["", f"Grad-CAM cases generated: {correct_cases} correct and {wrong_cases} incorrect.",\n              "Complete the saved review form for every case before finalizing the Day 4 backup.",\n              "Use validation evidence to propose one fix. Final held-out test comparisons are reserved for Day 5."]\n    (directory / "evaluation_report.md").write_text("\\n".join(lines) + "\\n", encoding="utf-8")\n    print(f"Analysis ready: {directory}\\nReview {correct_cases} correct and {wrong_cases} incorrect cases.", flush=True)\n    return directory\n\n\ndef package_day4(root, directory):\n    root, directory = Path(root).resolve(), Path(directory).resolve()\n    summary = json.loads((directory / "summary.json").read_text())\n    progress = review_progress(directory)\n    if summary["mode"] == "full" and (progress["remaining"] or progress["correct_reviewed"] < 20 or progress["incorrect_reviewed"] < 20):\n        raise ValueError("Complete the 20 correct and 20 incorrect cases in the review form before making the final backup.")\n    notes = json.loads((directory / "day4_observations.json").read_text())\n    if summary["mode"] == "full" and any("TODO" in str(value) for value in notes.values()):\n        raise ValueError("Fill the observation answers before making the final backup.")\n    summary["manual_review_complete"] = summary["mode"] == "full" and progress["remaining"] == 0\n    summary["review_progress"] = progress\n    write_json(directory / "summary.json", summary)\n    source_day3, _, _ = load_day3_results(root)\n    source_day2, _ = load_day2_results(root)\n    destination = root / ("day4_smoke_backup.zip" if summary["mode"] == "smoke" else "day4_backup.zip")\n    with zipfile.ZipFile(destination, "w", zipfile.ZIP_DEFLATED) as archive:\n        for name in ("flower_data.py", "flower_pipeline.py", "flower_training.py", "flower_transfer.py", "flower_evaluation.py", "flower_review.py",\n                     "outputs/day2_import.json", "outputs/day3_import.json"):\n            archive.write(root / name, name)\n        for folder in (root / "data/processed", root / "outputs/day1", source_day2, source_day3, directory):\n            for path in sorted(folder.rglob("*")):\n                if path.is_file() and ".matplotlib" not in path.parts:\n                    archive.write(path, path.relative_to(root))\n        credits = root / "data/raw/flower_photos/LICENSE.txt"\n        if credits.is_file():\n            archive.write(credits, "docs/DATASET_IMAGE_CREDITS.txt")\n    return destination\n', 'flower_review.py': '"""A small notebook form for recording the actual Grad-CAM case review."""\n\nimport html\nimport json\nfrom pathlib import Path\n\nfrom flower_evaluation import review_progress, save_case_review\n\n\ndef launch_review_form(directory):\n    import ipywidgets as widgets\n    from IPython.display import display\n\n    directory = Path(directory)\n    cases = json.loads((directory / "review_cases.json").read_text(encoding="utf-8"))\n    if not cases:\n        raise ValueError("No review cases were generated.")\n    incomplete = [i for i, case in enumerate(cases) if not (case.get("reviewed") and case.get("reviewer") == "human")]\n    state = {"index": incomplete[0] if incomplete else 0}\n    progress_label = widgets.HTML()\n    caption = widgets.HTML()\n    panel = widgets.Image(format="png", layout=widgets.Layout(width="100%", max_width="900px"))\n    attention = widgets.Dropdown(description="Heatmap:", options=[("Choose...", None), ("Mainly flower", "flower"),\n                                 ("Mainly background", "background"), ("Flower and background", "mixed"), ("Unclear / flat", "unclear")])\n    pattern = widgets.Dropdown(description="Pattern:", options=[("Choose...", None)], layout=widgets.Layout(width="95%"))\n    comment = widgets.Textarea(description="Note:", placeholder="Optional: describe the visible evidence or uncertainty.",\n                               layout=widgets.Layout(width="95%", height="80px"))\n    previous = widgets.Button(description="Previous")\n    save_next = widgets.Button(description="Save and next", button_style="primary")\n    message = widgets.HTML()\n\n    def render():\n        current = json.loads((directory / "review_cases.json").read_text(encoding="utf-8"))\n        case = current[state["index"]]\n        progress = review_progress(directory)\n        progress_label.value = (f"<b>Reviewed {progress[\'reviewed\']}/{progress[\'total\']}</b> "\n                                f"({progress[\'correct_reviewed\']} correct, {progress[\'incorrect_reviewed\']} incorrect). "\n                                f"Showing case {state[\'index\']+1}/{len(current)}.")\n        caption.value = (f"<b>{html.escape(case[\'case_id\'])}</b> — "\n                         f"{\'CORRECT\' if case[\'correct\'] else \'INCORRECT\'} prediction. "\n                         f"<br>{html.escape(case[\'relative_path\'])}")\n        panel.value = (directory / case["panel"]).read_bytes()\n        attention.value = case.get("attention") or None\n        choices = [("Choose...", None)]\n        if case["correct"]:\n            choices.append(("Correct prediction; no obvious issue", "correct_prediction"))\n        choices.extend([("Similar-looking classes", "class_confusion"), ("Small or distant flower", "small_subject"),\n                        ("Blur, lighting, or edited colour", "image_quality"), ("Multiple flowers or clutter", "clutter"),\n                        ("Possible label ambiguity", "label_ambiguity"), ("Possible background reliance", "background_candidate"),\n                        ("Unclear / other", "unclear_other")])\n        pattern.options = choices\n        valid_patterns = {value for _, value in choices}\n        pattern.value = case.get("pattern") if case.get("pattern") in valid_patterns else None\n        comment.value = case.get("comment", "")\n        previous.disabled = state["index"] == 0\n\n    def save_clicked(button):\n        if attention.value is None or pattern.value is None:\n            message.value = "<span style=\'color:#b42318\'>Choose a heatmap category and an observed pattern before saving.</span>"\n            return\n        current = json.loads((directory / "review_cases.json").read_text(encoding="utf-8"))\n        progress = save_case_review(directory, current[state["index"]]["case_id"], attention.value, pattern.value, comment.value)\n        if progress["remaining"]:\n            current = json.loads((directory / "review_cases.json").read_text(encoding="utf-8"))\n            later = [i for i in range(state["index"] + 1, len(current)) if not (current[i].get("reviewed") and current[i].get("reviewer") == "human")]\n            remaining = [i for i, case in enumerate(current) if not (case.get("reviewed") and case.get("reviewer") == "human")]\n            state["index"] = later[0] if later else remaining[0]\n            message.value = "Saved. Review the next case."\n        else:\n            message.value = "<b>All cases are reviewed. Continue to the observations cell, then make your backup.</b>"\n        render()\n\n    def previous_clicked(button):\n        state["index"] = max(0, state["index"] - 1)\n        message.value = "Previous case shown. Unsaved form changes were not recorded."\n        render()\n\n    save_next.on_click(save_clicked)\n    previous.on_click(previous_clicked)\n    render()\n    explanation = widgets.HTML("Look at the input crop, predicted-class heatmap, and true-class heatmap. "\n                               "Heatmaps suggest class sensitivity; they do not prove background bias or identify object boundaries. "\n                               "Softmax scores are uncalibrated. Choose the categories that match what you can see.")\n    form = widgets.VBox([progress_label, caption, panel, explanation, attention, pattern, comment,\n                         widgets.HBox([previous, save_next]), message])\n    display(form)\n    return form\n'}
for name, source in HELPER_SOURCES.items():
    destination = ROOT / name
    if IN_COLAB or not destination.exists():
        destination.write_text(source, encoding='utf-8')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import importlib
for module_name in ('flower_data', 'flower_pipeline', 'flower_training', 'flower_transfer', 'flower_evaluation', 'flower_review'):
    importlib.reload(importlib.import_module(module_name))
from flower_data import CLASS_NAMES, download_dataset, write_json
from flower_evaluation import (restore_day3_backup, load_day3_results, run_day4_analysis, review_progress, package_day4)
from flower_review import launch_review_form
import json
print('Day 4 helpers ready:', ROOT)

## 3. Restore Day 3

If asked, upload **`day3_backup (1).zip`**, or your latest completed Day 3 backup.
It restores all four checkpoints and the accepted Day 1 split manifest.
Flower photographs are downloaded again only when missing.

In [ ]:
if not (ROOT / "outputs/day3_import.json").is_file():
    if IN_COLAB:
        from google.colab import files
        print("Upload your latest completed Day 3 backup ZIP now.")
        uploaded = files.upload()
        backups = [name for name in uploaded if name.lower().endswith(".zip")]
        if len(backups) != 1:
            raise ValueError("Select exactly one Day 3 backup ZIP and rerun this cell.")
        restore_day3_backup(Path(backups[0]), ROOT)
    else:
        raise FileNotFoundError("First use restore_day3_backup(backup_path, ROOT) to restore your completed Day 3 backup.")
day3_directory, day3_comparison, checkpoint_paths = load_day3_results(ROOT)
download_dataset(ROOT / "data/raw")
print("Verified Day 3 results:", day3_directory)
print("Selected model:", day3_comparison["selected_by_validation"])
print("Saved checkpoints:", list(checkpoint_paths))

## 4. Evaluate the models and create the heatmaps

This evaluates all **551 validation images** with each saved model. Then it
creates 40 Grad-CAM panels for the selected model. Wait until **Analysis ready** appears.

Correct cases are sampled with seed 42 across true classes. Incorrect cases
are selected across true classes, taking higher-confidence errors first.
The gallery is a diagnostic sample; the metrics above cover the full validation set.

Running this cell again starts a new analysis folder. Once it finishes, continue
below; later edits to notes only require rerunning the notes and backup cells.

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 2 if IN_COLAB else 0
CASES_PER_OUTCOME = 20
SEED = 42

analysis = run_day4_analysis(ROOT, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                            per_outcome=CASES_PER_OUTCOME, seed=SEED)
summary = json.loads((analysis / "summary.json").read_text())
if not summary["review_target_met"]:
    raise RuntimeError("Fewer than 20 correct or 20 incorrect predictions are available. Keep these results and discuss the shortfall before finalizing.")

## 5. Read the metrics

**Precision:** among images predicted as a class, how many were correct?
**Recall:** among actual images of a class, how many were found?
**F1:** balances precision and recall. **Macro-F1** gives each class equal weight.
Confusion matrix rows are true labels; columns are predicted labels.

All four models' full per-class reports and prediction tables are saved in the backup.

In [ ]:
from IPython.display import display, Markdown, Image as DisplayImage

rows = summary["model_metrics"]
table = ["| Model | Val accuracy | Macro-F1 | Errors |", "|---|---:|---:|---:|"]
table += [f"| {row['run']} | {row['accuracy']:.2%} | {row['macro_f1']:.4f} | {row['incorrect']} |" for row in rows]
display(Markdown("\n".join(table)))
selected_name = summary["selected_model"]
report = json.loads((analysis / selected_name / "classification_report.json").read_text())
table = [f"### Per-class report: {selected_name}", "", "| Class | Precision | Recall | F1 | Support |",
         "|---|---:|---:|---:|---:|"]
table += [f"| {name} | {report[name]['precision']:.3f} | {report[name]['recall']:.3f} | {report[name]['f1-score']:.3f} | {report[name]['support']} |" for name in CLASS_NAMES]
display(Markdown("\n".join(table)))
display(DisplayImage(filename=str(analysis / "per_class_f1.png")))
for row in rows:
    display(DisplayImage(filename=str(analysis / row["run"] / "confusion_matrix.png")))
failures = json.loads((analysis / "failure_patterns.json").read_text())
print("Lowest F1 class:", failures["lowest_f1_class"])
print("Largest confusion pairs:")
for pair in failures["confusion_pairs"][:5]:
    print(f"  True {pair['true_class']} -> predicted {pair['predicted_class']}: {pair['count']} images")

## 6. Review the 40 Grad-CAM cases - pause here

Every panel shows the **actual input crop**, the **predicted-class** heatmap,
and the **true-class** heatmap. For a correct prediction, the two heatmaps match.
Brighter colours show relatively stronger positive sensitivity to the selected
class logit. Each map is normalized separately; colour strength cannot be compared
numerically between maps. Flat maps contain no positive spatial contrast.

Inspect where the heatmap falls: flower, background, both, or unclear. Choose an
observed pattern, optionally add a note, then click **Save and next**.
Continue until **Reviewed 40/40 (20 correct, 20 incorrect)**.

Heatmaps are approximate explanations, not object boundaries or proof of a cause.
A background hotspot is a candidate concern; it does not by itself prove background
bias. Softmax scores are uncalibrated. Avoid inventing an explanation when unclear.

Review progress is saved after each click. Rerunning this form cell resumes the
first unfinished case. Do not rerun the analysis cell to resume reviewing.

In [ ]:
if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()
review_form = launch_review_form(analysis)

## 7. Check your review and summarize the patterns

Run this after the form reaches 40/40. The counts below summarize your recorded
observations. They describe this deliberately selected sample, not population rates.

In [ ]:
from collections import Counter

progress = review_progress(analysis)
print(progress)
if progress["remaining"] or progress["correct_reviewed"] < 20 or progress["incorrect_reviewed"] < 20:
    raise ValueError("Return to the form and finish all 20 correct + 20 incorrect cases, then rerun this cell.")
cases = json.loads((analysis / "review_cases.json").read_text())
wrong_cases = [case for case in cases if not case["correct"]]
attention_counts = dict(Counter(case["attention"] for case in wrong_cases))
pattern_counts = dict(Counter(case["pattern"] for case in wrong_cases))
print("Incorrect-case heatmap categories:", attention_counts)
print("Incorrect-case observed patterns:", pattern_counts)

## 8. Write two observation answers

Replace both TODO strings with your own evidence, then run this cell.
Mention recurring confusions and what you saw in the incorrect cases. Propose
**one** feasible fix for Day 5 and explain which validation evidence supports it.
For example, a crop/scale augmentation change may help small-subject errors;
such a change is a hypothesis to test, not a guaranteed improvement.

Keep labels and splits fixed. Do not choose a fix using test-set errors.
If you want help interpreting your results, share `day4_observations.json` and
your metrics/review outputs. You can download this JSON before making the final backup.

In [ ]:
selected_metrics = next(row for row in rows if row["run"] == selected_name)
my_observations = {
    "analysis_split": "val",
    "selected_model": selected_name,
    "validation_accuracy": selected_metrics["accuracy"],
    "validation_macro_f1": selected_metrics["macro_f1"],
    "lowest_f1_class": failures["lowest_f1_class"],
    "top_confusion_pairs": failures["confusion_pairs"][:5],
    "review_progress": progress,
    "incorrect_attention_counts": attention_counts,
    "incorrect_pattern_counts": pattern_counts,
    "failure_patterns_with_evidence": "TODO: describe recurring errors and visible heatmap patterns, with case IDs or counts; distinguish observations from possible causes",
    "one_proposed_fix_and_reason": "TODO: choose one specific change for Day 5 and explain which validation evidence supports testing it",
    "limitations": "One split and seed; review errors selected partly by confidence; Grad-CAM is approximate and independently normalized; softmax scores are uncalibrated; no test metrics yet.",
}
write_json(analysis / "day4_observations.json", my_observations)
print("Saved:", analysis / "day4_observations.json")
if any("TODO" in str(value) for value in my_observations.values()):
    print("Replace both TODO answers and rerun this cell before making your final backup.")

## 9. Download your completed Day 4 backup

This cell checks that all 40 cases have been reviewed and both observation answers
are filled. It includes the four checkpoints, original splits, earlier results,
new metrics/plots, all heatmaps, the standalone HTML gallery, and your review notes.

Keep **`day4_backup.zip`**. Also download this executed notebook using the top
**File -> Download -> Download .ipynb** menu. The notebook download is separate
from the left Files panel and separate from the ZIP download.

In [ ]:
exact_versions = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(analysis / "requirements-runtime.txt").write_text(exact_versions, encoding="utf-8")
backup = package_day4(ROOT, analysis)
print("Backup:", backup)
if IN_COLAB:
    from google.colab import files
    files.download(str(backup))

## Next: Day 5

Test one proposed fix using validation, then run the final held-out test comparison,
create an upload demo, and prepare the report/presentation.

Sources:
- [Grad-CAM paper](https://openaccess.thecvf.com/content_ICCV_2017/papers/Selvaraju_Grad-CAM_Visual_Explanations_ICCV_2017_paper.pdf)
- [PyTorch autograd](https://docs.pytorch.org/docs/stable/autograd.html)
- [Classification report definitions](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)

The helper computes these report metrics directly from confusion counts with zero
division set to zero; scikit-learn is not required. Image credits are retained in the backup.